## [title]

## [setup_head]

In [ ]:
import glob
import os
import pickle as pkl
import re
import sys
import time
from collections import Counter
from itertools import combinations, zip_longest

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from IPython.display import display
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem, Draw, rdFMCS
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog('rdApp.*')
REPO = os.path.abspath('..')
sys.path[:0] = [REPO, os.path.join(REPO, 'src')]   # the checkpoint pickle imports `mist` from src/

# Pickles written under numpy 2 reference numpy._core; alias it on numpy 1.x.
if not hasattr(np, '_core'):
    import numpy.core as _np_core
    sys.modules.setdefault('numpy._core', _np_core)
    for _sub in ('multiarray', 'numeric', '_multiarray_umath'):
        sys.modules.setdefault(f'numpy._core.{_sub}', getattr(_np_core, _sub))

# ---- paths -----------------------------------------------------------------
RUN_TMS   = 'data/yee_new_0810/tms/outputs/2026-10-07/22-28-16-inference'
RUN_NOTMS = 'data/yee_new_0810/notms/outputs/2026-10-07/11-08-22-inference'
REF_CSV   = 'data/yee_new_0810/reference_inchi.csv'
YEE_DIR   = '../data/neims/yee/mist_inputs'
TRAIN_DIR = '../data/neims/gecko_tms_mixed_augment_tms_atmomaccs_tms_test/mist_inputs'
CKPT      = '../data/checkpoints/yee_tms/tms_model.ckpt'
# Results of the model runs in this notebook. Delete a file to recompute that experiment.
CACHE_DIR = 'data/yee_new_0810/tms/walkthrough_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# ---- the worked example ----------------------------------------------------
CATECHOL, HYDROQUINONE = 'LDY-20562', 'LDY-20550'
EXAMPLE_FORMULA = 'C12H22O2Si2'
ORTHO = 'C[Si](C)(C)Oc1ccccc1O[Si](C)(C)C'
META  = 'C[Si](C)(C)Oc1cccc(O[Si](C)(C)C)c1'
PARA  = 'C[Si](C)(C)Oc1ccc(O[Si](C)(C)C)cc1'
ISOMERS = {'ortho (catechol)': ORTHO, 'meta (resorcinol)': META, 'para (hydroquinone)': PARA}

# ---- knobs -----------------------------------------------------------------
TOP_K_LIST = [1, 3, 5, 10, 20, 50, 100]
MORGAN_R, MORGAN_NBITS = 2, 2048    # configs/dataset/yee.yaml: what the encoder was pretrained to predict
LIVE_BATCH = 16                     # samples per decoder batch; about 2.5 GB of RAM at this size
LIVE_THREADS = 10
SEED = 7

# ---- plot theme (same palette as test_set_overview.ipynb) ------------------
C1, C2, C3, C4 = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
NEUTRAL = '#b4b4b0'
INK, MUTED = '#222222', '#666666'
GRID_KW = dict(color='0.88', linewidth=0.6)
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5, 'legend.fontsize': 8.5, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.titlelocation': 'left',
    'axes.titleweight': 'bold', 'figure.dpi': 120, 'axes.axisbelow': True,
})


def cached(name, fn):
    """Load CACHE_DIR/<name>.pkl, or run fn() once and store its result."""
    path = os.path.join(CACHE_DIR, f'{name}.pkl')
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pkl.load(f)
    t0 = time.time()
    out = fn()
    with open(path, 'wb') as f:
        pkl.dump(out, f)
    print(f'  computed {name} in {(time.time() - t0) / 60:.1f} min -> {path}')
    return out


def key_2d(inchi):
    """InChIKey of the 2D structure: stereo layers dropped, as generated molecules carry no stereo."""
    return Chem.InchiToInchiKey(re.sub(r'/[btms][^/]*', '', inchi))


def morgan(mol):
    return AllChem.GetMorganFingerprintAsBitVect(mol, MORGAN_R, nBits=MORGAN_NBITS)


def wilson(k, n, z=1.96):
    """Wilson 95% interval for k successes in n trials."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (max(0.0, c - h), min(1.0, c + h))

## [load_head]

In [ ]:
def load_run(run_dir):
    """Candidate table of one inference run, ranked by generation count as in K_ACC_Collection."""
    c = pd.read_pickle(os.path.join(run_dir, 'candidates_cache.pkl'))
    c['key_2d'] = c['inchi'].map(key_2d)
    # Stable sort keeps first-generated order among ties, like Counter.most_common
    return c.sort_values('count', ascending=False, kind='stable').reset_index(drop=True)


cand, cand_notms = load_run(RUN_TMS), load_run(RUN_NOTMS)
by_uid = {u: g for u, g in cand.groupby('uid', sort=False)}
summary = pd.read_csv(os.path.join(RUN_TMS, 'results_summary.csv')).set_index('uid')

names = np.loadtxt(os.path.join(RUN_TMS, 'batch_names.txt'), dtype=np.str_).tolist()
Y = torch.load(os.path.join(RUN_TMS, 'predicted_fingerprints.pt')).float().numpy()   # the encoder's output
y_of = dict(zip(names, Y))
Y_MEAN = Y.mean(0)

labels = pd.read_csv(f'{YEE_DIR}/labels.tsv', sep='\t').set_index('spec')
spec_df = pd.read_pickle(f'{YEE_DIR}/df_neims_yee_3_9_22.pkl').set_index('name')
with open(f'{YEE_DIR}/yee_subforms_3_9_22.pkl', 'rb') as f:
    subforms = pkl.load(f)

refs = pd.read_csv(REF_CSV, dtype=str, keep_default_na=False).query("status == 'found'").set_index('uid')
UNCERTAIN_NOTE = re.compile(r'tentative|possibly|could', re.I)

N_SAMPLES = int(summary['num_samples'].iloc[0])
print(f'{len(names)} spectra x {N_SAMPLES} generated candidates; '
      f'{labels["identified"].sum()} named, {(~labels["identified"]).sum()} unidentified in the library')
print(f'{len(refs)} spectra have a reference structure from NIST '
      f'({sum(bool(UNCERTAIN_NOTE.search(n)) for n in refs["note"])} marked uncertain)')
print(f'instrument: {labels["instrument"].unique().tolist()}, derivatization: {labels["derivatization"].unique().tolist()}')

## [s1_head]

In [ ]:
def box(ax, x, y, w, h, title, body, fill):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.008,rounding_size=0.012',
                                facecolor=fill, edgecolor='0.55', linewidth=0.8))
    ax.text(x + w / 2, y + h - 0.035, title, ha='center', va='top', fontsize=9.5, fontweight='bold', color=INK)
    ax.text(x + w / 2, y + h - 0.125, body, ha='center', va='top', fontsize=8, color=INK, linespacing=1.35)


def arrow(ax, a, b, style='-|>', **kw):
    ax.add_patch(FancyArrowPatch(a, b, arrowstyle=style, mutation_scale=11, color=MUTED, linewidth=1.1, **kw))


fig, ax = plt.subplots(figsize=(12.5, 3.5))
ax.set(xlim=(0, 1), ylim=(0, 1))
ax.axis('off')
data_fill, net_fill = '#f1f1ee', '#dbe8f8'
stages = [
    (0.005, 'EI spectrum', 'integer m/z, intensity\n+ molecular formula\n(given, not inferred)', data_fill),
    (0.205, 'Peak tokens', 'top 50 peaks\neach m/z → a subformula\nof the parent, sqrt-scaled\nthen log intensity', data_fill),
    (0.405, 'Encoder', 'set transformer over tokens\n(MIST FormulaTransformer)\n23.4 M parameters\n+ 8.4 M 4096 → 2048 head', net_fill),
    (0.605, 'Decoder', 'graph transformer, 53.2 M\n500 reverse diffusion steps\nheavy atoms fixed,\nonly bonds are sampled', net_fill),
    (0.805, 'Ranked list', 'run 100 times\ndrop invalid, merge by InChI\nrank by count', data_fill),
]
W, H, Y0 = 0.19, 0.60, 0.33
for x, title, body, fill in stages:
    box(ax, x, Y0, W, H, title, body, fill)
for (x, *_), (x2, *_) in zip(stages, stages[1:]):
    arrow(ax, (x + W + 0.004, Y0 + H / 2), (x2 - 0.004, Y0 + H / 2))
ax.text(0.6, Y0 + H / 2 + 0.03, 'y', ha='center', fontsize=10, style='italic', color=INK)
# The formula reaches the decoder directly, as the atom set of the graph
ax.plot([0.10, 0.10, 0.70], [Y0 - 0.01, 0.13, 0.13], color=MUTED, linewidth=1.1)
arrow(ax, (0.70, 0.13), (0.70, Y0 - 0.01))
ax.text(0.40, 0.155, 'the formula sets the heavy atoms directly; hydrogens are not modelled', ha='center',
        fontsize=8.5, color=MUTED)
ax.add_patch(plt.Rectangle((0.005, 0.015), 0.018, 0.05, facecolor=net_fill, edgecolor='0.55', linewidth=0.8))
ax.text(0.03, 0.04, 'learned', va='center', fontsize=8.5, color=MUTED)
ax.add_patch(plt.Rectangle((0.095, 0.015), 0.018, 0.05, facecolor=data_fill, edgecolor='0.55', linewidth=0.8))
ax.text(0.12, 0.04, 'fixed preprocessing / postprocessing', va='center', fontsize=8.5, color=MUTED)
plt.show()

## [s1_ml]

## [s2_head]

In [ ]:
# Which spectra the model learned from. Reads the 80 MB labels file; takes a few seconds.
train_labels = pd.read_csv(f'{TRAIN_DIR}/labels.tsv', sep='\t',
                           usecols=['dataset', 'spec', 'smiles', 'inchikey', 'instrument'])
train_split = pd.read_csv(f'{TRAIN_DIR}/split_random_guarded.tsv', sep='\t').rename(columns={'name': 'spec'})
train_labels = train_labels.merge(train_split, on='spec')

print('instrument values across all splits:', train_labels['instrument'].unique().tolist())
composition = train_labels.groupby(['dataset', 'split']).size().unstack(fill_value=0)
display(composition)

MOTIFS = {'aromatic ring': 'a', 'TMS ester': 'C(=O)O[Si](C)(C)C', 'aryl O-TMS': 'c-O[Si](C)(C)C',
          'alkyl O-TMS': '[CX4]O[Si](C)(C)C', 'peroxy O-O-TMS': 'OO[Si](C)(C)C'}
MOTIF_Q = {k: Chem.MolFromSmarts(v) for k, v in MOTIFS.items()}


def motif_rates(smiles):
    mols = [m for m in map(Chem.MolFromSmiles, smiles) if m is not None]
    return {k: np.mean([m.HasSubstructMatch(q) for m in mols]) for k, q in MOTIF_Q.items()}, mols


rate_rows, heavy = {}, {}
_train = train_labels[train_labels['split'] == 'train']
for ds, disp in (('gecko_tms_neims', 'GECKO-A (TMS)'), ('mixed_augment_tms_neims', 'Augmentation (TMS)')):
    sample = _train[_train['dataset'] == ds].sample(4000, random_state=0)['smiles']
    rate_rows[disp], _m = motif_rates(sample)
    heavy[disp] = np.median([m.GetNumHeavyAtoms() for m in _m])
ref_smiles = [Chem.MolToSmiles(Chem.MolFromInchi(i)) for i in refs['inchi']]
rate_rows['Yee references (real spectra)'], _m = motif_rates(ref_smiles)
heavy['Yee references (real spectra)'] = np.median([m.GetNumHeavyAtoms() for m in _m])
motif_table = pd.DataFrame(rate_rows)

fig, ax = plt.subplots(figsize=(8.2, 3.1))
ypos = np.arange(len(motif_table))
for i, (col, color) in enumerate(zip(motif_table.columns, (C1, C2, C3))):
    ax.barh(ypos + (i - 1) * 0.26, motif_table[col] * 100, height=0.23, color=color, label=col)
ax.set_yticks(ypos, motif_table.index)
ax.invert_yaxis()
ax.set_xlabel('Share of molecules containing the motif (%)')
ax.set_title('What the model was trained on, against what it is asked to identify')
ax.grid(axis='x', **GRID_KW)
ax.tick_params(axis='y', length=0)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()
print('median heavy atoms:', {k: float(v) for k, v in heavy.items()})

## [s2_tail]

## [s3_head]

In [ ]:
def peak_table(uid):
    """What the encoder receives for one spectrum: one row per token, in token order, with its pooling weight."""
    o = subforms[uid]['output_tbl']
    t = pd.DataFrame({'mz': np.array(o['mz'], dtype=int), 'token': o['formula'],
                      'token_mass': o['mono_mass'], 'scaled_inten': o['ms2_inten']})
    # FormulaTransformer._pool_out, set_pooling='intensity' on inten_transform='log' values. The zero-intensity
    # CLS token sits in the same sum: log(1e-5) is its value.
    log_i = np.r_[np.log(t['scaled_inten'].to_numpy() + 1e-5), np.log(1e-5)]
    t['pool_weight'] = (log_i / log_i.sum())[:-1]
    return t


def raw_spectrum(uid):
    s = np.asarray(spec_df.loc[uid, 'spec'], dtype=float)
    return s[:, 0], s[:, 1] / s[:, 1].max()


def mirror_plot(uid_a, uid_b, label_a, label_b, n_label=7):
    """Head-to-tail spectrum. Coloured sticks reach the encoder; grey ones are cut by the 50-peak limit."""
    fig, ax = plt.subplots(figsize=(10.5, 4.0))
    for uid, sign, color, label in ((uid_a, 1, C1, label_a), (uid_b, -1, C2, label_b)):
        mz, inten = raw_spectrum(uid)
        kept = np.isin(mz, peak_table(uid)['mz'])
        ax.vlines(mz[~kept], 0, sign * inten[~kept] * 100, color=NEUTRAL, linewidth=0.8)
        ax.vlines(mz[kept], 0, sign * inten[kept] * 100, color=color, linewidth=1.1, label=label)
        for i in np.argsort(-inten)[:n_label]:
            ax.text(mz[i], sign * (inten[i] * 100 + 2), f'{mz[i]:.0f}', ha='center',
                    va='bottom' if sign > 0 else 'top', fontsize=8, color=INK)
    ax.axhline(0, color='0.3', linewidth=0.7)
    ax.set_ylim(-118, 118)
    ax.set_yticks([-100, -50, 0, 50, 100], [100, 50, 0, 50, 100])
    ax.set_xlabel('m/z')
    ax.set_ylabel('Relative intensity (%)')
    ax.grid(axis='y', **GRID_KW)
    ax.legend(loc='upper right', ncol=1, title=None)
    return fig, ax


fig, ax = mirror_plot(CATECHOL, HYDROQUINONE, 'catechol 2TMS (LDY-20562)', 'hydroquinone 2TMS (LDY-20550)')
ax.set_title('Two isomers, same formula C12H22O2Si2')
plt.tight_layout()
plt.show()

## [s3_tokens]

In [ ]:
MONO = {'C': 12.0, 'H': 1.00782503, 'O': 15.99491462, 'Si': 27.97692653, 'N': 14.0030740}


def mono_mass(formula):
    return sum(MONO[e] * (int(n) if n else 1) for e, n in re.findall(r'([A-Z][a-z]?)(\d*)', formula))


# The ions a spectroscopist would write for these peaks, against the token the pipeline assigned.
DIAGNOSTIC = {73: ('C3H9Si', 'trimethylsilyl cation'), 75: ('C2H7OSi', 'dimethylsilanol ion'),
              147: ('C5H15OSi2', 'pentamethyldisiloxanyl ion'), 239: ('C11H19O2Si2', '[M - CH3]+'),
              254: ('C12H22O2Si2', 'M+.')}
t = peak_table(CATECHOL).set_index('mz')
display(pd.DataFrame([{'m/z': mz, 'expected ion': ion, 'expected formula': f, 'expected mass': mono_mass(f),
                       'assigned token': t.loc[mz, 'token'], 'token mass': t.loc[mz, 'token_mass'],
                       'scaled intensity': t.loc[mz, 'scaled_inten'], 'pooling weight': t.loc[mz, 'pool_weight']}
                      for mz, (f, ion) in DIAGNOSTIC.items()]).round(4).style.hide(axis='index'))

# Across the whole library
tok73 = {True: Counter(), False: Counter()}
n_raw, n_kept, inten_kept = [], [], []
for u in names:
    o = subforms[u]['output_tbl']
    mz, inten = raw_spectrum(u)
    n_raw.append(len(mz))
    n_kept.append(len(o['mz']))
    inten_kept.append(inten[np.isin(mz, o['mz'])].sum() / inten.sum())
    if 73.0 in o['mz']:
        tok73['N' in subforms[u]['cand_form']][o['formula'][o['mz'].index(73.0)]] += 1
print(f'peaks per spectrum: median {np.median(n_raw):.0f} measured -> {np.median(n_kept):.0f} kept; '
      f'kept peaks carry a median {np.median(inten_kept):.0%} of total ion intensity (min {np.min(inten_kept):.0%})')
print('token given to m/z 73 when the parent has no N:', tok73[False].most_common(3))
print('token given to m/z 73 when the parent has N:   ', tok73[True].most_common(4))

## [s3_tail]

## [s4_head]

In [ ]:
# y against the true Morgan bits, and against a vector that knows nothing about the spectrum
auc_y, auc_const, labelled = [], [], []
for uid, r in refs.iterrows():
    bits = np.zeros(MORGAN_NBITS, dtype=int)
    DataStructs.ConvertToNumpyArray(morgan(Chem.MolFromInchi(r['inchi'])), bits)
    auc_y.append(roc_auc_score(bits, y_of[uid]))
    auc_const.append(roc_auc_score(bits, Y_MEAN))
auc_y, auc_const = np.array(auc_y), np.array(auc_const)

# Do repeat measurements of one compound land together? Centred, because all y share a large common component.
Z = {u: (y_of[u] - Y_MEAN) / np.linalg.norm(y_of[u] - Y_MEAN) for u in refs.index}
smiles_2d = {u: Chem.MolToSmiles(Chem.MolFromInchi(r['inchi']), isomericSmiles=False) for u, r in refs.iterrows()}
pairs = {'same compound\n(repeat spectra)': [], 'same formula,\ndifferent isomer': [], 'unrelated': []}
for a, b in combinations(refs.index, 2):
    sim = float(Z[a] @ Z[b])
    if smiles_2d[a] == smiles_2d[b]:
        pairs['same compound\n(repeat spectra)'].append(sim)
    elif refs.loc[a, 'formula'] == refs.loc[b, 'formula']:
        pairs['same formula,\ndifferent isomer'].append(sim)
    else:
        pairs['unrelated'].append(sim)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.5), gridspec_kw={'width_ratios': [1, 1.25]})
ax = axes[0]
ax.plot([0.45, 1], [0.45, 1], color=NEUTRAL, linewidth=0.9)
ax.scatter(auc_const, auc_y, s=16, color=C1, alpha=0.75, linewidths=0)
ax.set(xlim=(0.45, 1), ylim=(0.45, 1), xlabel='AUROC of a constant vector (mean y over all spectra)',
       ylabel='AUROC of this spectrum\'s y')
ax.set_title('y as a fingerprint: barely above a constant')
ax.grid(**GRID_KW)

ax = axes[1]
rng = np.random.default_rng(0)
for i, (k, v) in enumerate(pairs.items()):
    v = np.array(v)
    show_v = rng.choice(v, min(len(v), 400), replace=False)
    ax.scatter(show_v, i + rng.uniform(-0.17, 0.17, len(show_v)), s=11, color=C1, alpha=0.55, linewidths=0)
    ax.plot([np.median(v)] * 2, [i - 0.3, i + 0.3], color=INK, linewidth=1.8)
    ax.text(1.03, i, f'median {np.median(v):.2f}\nn = {len(v):,}', va='center', fontsize=8, color=MUTED)
ax.set_yticks(range(len(pairs)), list(pairs))
ax.invert_yaxis()
ax.set(xlim=(-0.45, 1.0), xlabel='Cosine similarity of centred y')
ax.set_title('y as an embedding: same compound, same place')
ax.grid(axis='x', **GRID_KW)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
plt.show()
print(f'mean per-spectrum AUROC: y {auc_y.mean():.3f}, constant {auc_const.mean():.3f}; '
      f'y wins on {np.mean(auc_y > auc_const):.0%} of spectra')

## [s4_live_head]

In [ ]:
_live = {}


def live():
    """Build the model from the checkpoint once, on first use. Nothing above this needs it."""
    if _live:
        return _live
    from hydra import compose, initialize_config_dir
    from hydra.core.global_hydra import GlobalHydra
    from src.analysis.visualization import MolecularVisualization
    from src.datasets.inference_dataset import InferenceDataModule, InferenceDatasetInfos
    from src.diffusion.extra_features import ExtraFeatures
    from src.diffusion.extra_features_molecular import ExtraMolecularFeatures
    from src.diffusion_model_spec2mol import Spec2MolDenoisingDiffusion
    from src.metrics.molecular_metrics_discrete import TrainMolecularMetricsDiscrete

    torch.set_num_threads(LIVE_THREADS)
    ck = torch.load(CKPT, map_location='cpu', weights_only=False, mmap=True)
    model_cfg = ck['hyper_parameters']['cfg']
    cwd = os.getcwd()
    os.chdir(REPO)      # configs/dataset/yee.yaml paths are repo-relative
    try:
        GlobalHydra.instance().clear()
        with initialize_config_dir(version_base='1.3', config_dir=os.path.join(REPO, 'configs')):
            cfg = compose(config_name='config_inference', overrides=[
                'dataset.stats_dir=' + model_cfg.dataset.stats_dir.replace('../../../', ''),
                'train.eval_batch_size=1', 'train.num_workers=0'])
        dm = InferenceDataModule(cfg)
        infos = InferenceDatasetInfos(dm, cfg)
        domain = ExtraMolecularFeatures(dataset_infos=infos)
        extra = ExtraFeatures(cfg.model.extra_features, dataset_info=infos)
        infos.compute_input_output_dims(datamodule=dm, extra_features=extra, domain_features=domain)
        model = Spec2MolDenoisingDiffusion(
            model_cfg, dataset_infos=infos, train_metrics=TrainMolecularMetricsDiscrete(infos),
            visualization_tools=MolecularVisualization(cfg.dataset.remove_h, dataset_infos=infos),
            extra_features=extra, domain_features=domain, load_pretrained_weights=False)
        model.load_state_dict(ck['state_dict'], strict=True)
    finally:
        os.chdir(cwd)
    del ck
    ds = dm.test_dataset
    _live.update(model=model.eval(), ds=ds, collate=dm.test_dataloader().collate_fn,
                 index={n: i for i, n in enumerate(ds.get_spectra_names())})
    return _live


def spectrum_batch(uid):
    L = live()
    return L['collate']([L['ds'][L['index'][uid]]])


@torch.no_grad()
def encode(batch):
    """Spectrum batch -> y, as predict_fingerprint in src/inference.py does for merge=downproject_4096."""
    m = live()['model']
    return m.merge_function(m.encoder(batch)[0])


def batch_from_tokens(base, form_vec, scaled_inten):
    """A one-spectrum batch with these peaks. form_vec and scaled_inten exclude the trailing CLS token."""
    n = len(scaled_inten)
    return {**base,
            'types': torch.tensor([[0] * n + [3]]),
            'form_vec': torch.cat([form_vec, torch.zeros(1, form_vec.shape[1])])[None],
            'ion_vec': torch.full((1, n + 1), float(base['ion_vec'][0, 0])),
            'intens': torch.log(torch.cat([scaled_inten, torch.zeros(1)]) + 1e-5)[None].float(),
            'num_peaks': torch.tensor([n + 1])}


def tokens_of(uid):
    b = spectrum_batch(uid)
    n = int(b['num_peaks'][0]) - 1      # the last token is CLS
    return b, b['form_vec'][0, :n], torch.exp(b['intens'][0, :n]) - 1e-5


def occlusion(uid):
    """Delete each peak in turn and measure how far y moves."""
    b, forms, inten = tokens_of(uid)
    y0 = encode(batch_from_tokens(b, forms, inten))[0].numpy()
    z0 = y0 - Y_MEAN
    rows = []
    for i in range(len(inten)):
        keep = torch.arange(len(inten)) != i
        z = encode(batch_from_tokens(b, forms[keep], inten[keep]))[0].numpy() - Y_MEAN
        rows.append(1 - float(z @ z0 / (np.linalg.norm(z) * np.linalg.norm(z0))))
    t = peak_table(uid).assign(moved=rows)
    return {'table': t, 'y_error': float(np.abs(y0 - y_of[uid]).max())}


occ = {u: cached(f'occlusion_{u}', lambda u=u: occlusion(u)) for u in (CATECHOL, HYDROQUINONE)}
print('rebuilt tokens reproduce the stored y of the run to within',
      f'{max(o["y_error"] for o in occ.values()):.1e}')

fig, axes = plt.subplots(2, 1, figsize=(10.5, 4.6), sharex=True)
for ax, u, color, title in ((axes[0], CATECHOL, C1, 'catechol 2TMS'), (axes[1], HYDROQUINONE, C2, 'hydroquinone 2TMS')):
    t = occ[u]['table']
    ax.vlines(t['mz'], 0, t['moved'], color=color, linewidth=1.3)
    for _, r in t.nlargest(6, 'moved').iterrows():
        ax.text(r['mz'], r['moved'], f"{r['mz']:.0f}", ha='center', va='bottom', fontsize=8, color=INK)
    ax.set_ylim(0, t['moved'].max() * 1.2)
    ax.set_ylabel('1 - cos(y, y without peak)')
    ax.set_title(title)
    ax.grid(axis='y', **GRID_KW)
axes[1].set_xlabel('m/z of the deleted peak')
plt.tight_layout()
plt.show()
for u, name in ((CATECHOL, 'catechol'), (HYDROQUINONE, 'hydroquinone')):
    t = occ[u]['table']
    print(f"{name}: top peaks by influence {t.nlargest(6, 'moved')['mz'].tolist()}, "
          f"by intensity {t.nlargest(6, 'scaled_inten')['mz'].tolist()}; "
          f"Spearman(influence, intensity) = {t['moved'].corr(t['scaled_inten'], method='spearman'):.2f}")

## [s4_tail]

## [s5_head]

In [ ]:
from src.diffusion.noise_schedule import MarginalUniformTransition, PredefinedNoiseScheduleDiscrete

T = 500
BOND_NAMES = ['no bond', 'single', 'double', 'triple', 'aromatic']
schedule = PredefinedNoiseScheduleDiscrete('cosine', T)
edge_marginals = torch.tensor(np.loadtxt(f'{TRAIN_DIR}/stats/edge_types.txt')).float()
edge_marginals = edge_marginals / edge_marginals.sum()
atom_marginals = torch.tensor(np.loadtxt(f'{TRAIN_DIR}/stats/atom_types.txt')).float()
transition = MarginalUniformTransition(atom_marginals / atom_marginals.sum(), edge_marginals, y_classes=0)
alpha_bar = schedule.get_alpha_bar(t_int=torch.arange(T + 1)).numpy()

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.7), gridspec_kw={'width_ratios': [1.25, 1]})
axes[0].plot(np.arange(T + 1), alpha_bar, color=C1, linewidth=2)
axes[0].set(xlabel='diffusion step t', ylabel='P(bond state left unchanged by noise)\nplus marginal resampling', xlim=(0, T), ylim=(0, 1.02))
axes[0].set_title('Noise schedule (cosine): ᾱ_t')
axes[0].grid(**GRID_KW)
axes[1].barh(BOND_NAMES, edge_marginals.numpy() * 100, color=C1, height=0.6)
for i, v in enumerate(edge_marginals.numpy()):
    axes[1].text(v * 100 + 1.5, i, f'{v:.1%}', va='center', fontsize=8.5, color=INK)
axes[1].invert_yaxis()
axes[1].set(xlim=(0, 110), xlabel='Share of atom pairs in training molecules (%)')
axes[1].set_title('Where the noise converges: edge marginals')
axes[1].tick_params(axis='y', length=0)
plt.tight_layout()
plt.show()

In [ ]:
BOND_STYLE = {1: dict(linewidth=1.1, linestyle='-'), 2: dict(linewidth=2.6, linestyle='-'),
              3: dict(linewidth=3.6, linestyle='-'), 4: dict(linewidth=1.6, linestyle=(0, (2, 1.2)))}
ATOM_COLOR = {'C': '#8a8a86', 'O': C2, 'Si': C1}


def draw_graph(ax, symbols, E, pos, title=None):
    """A bond matrix drawn on fixed atom positions, so noise shows as wrong edges rather than a moving layout."""
    n = len(symbols)
    for i in range(n):
        for j in range(i + 1, n):
            if E[i, j] > 0:
                ax.plot(pos[[i, j], 0], pos[[i, j], 1], color='0.25', zorder=1, solid_capstyle='round',
                        **BOND_STYLE[int(E[i, j])])
    for s in set(symbols):
        idx = [i for i, a in enumerate(symbols) if a == s]
        ax.scatter(pos[idx, 0], pos[idx, 1], s=46 if s != 'C' else 26, color=ATOM_COLOR.get(s, C4),
                   edgecolor='white', linewidth=0.8, zorder=2, label=s)
    ax.set_aspect('equal')
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=9, fontweight='normal', loc='center')


def layout(mol):
    m = Chem.Mol(mol)
    AllChem.Compute2DCoords(m)
    return m.GetConformer().GetPositions()[:, :2]


def bond_matrix(mol):
    E = np.zeros((mol.GetNumAtoms(), mol.GetNumAtoms()), dtype=np.uint8)
    code_of = {Chem.BondType.SINGLE: 1, Chem.BondType.DOUBLE: 2, Chem.BondType.TRIPLE: 3, Chem.BondType.AROMATIC: 4}
    for b in mol.GetBonds():
        E[b.GetBeginAtomIdx(), b.GetEndAtomIdx()] = E[b.GetEndAtomIdx(), b.GetBeginAtomIdx()] = code_of[b.GetBondType()]
    return E


def noised(E0, t, rng):
    """Sample E_t ~ q(E_t | E_0) with the model's own transition matrices, keeping the matrix symmetric."""
    Q = transition.get_Qt_bar(schedule.get_alpha_bar(t_int=torch.tensor([[t]])), 'cpu').E[0].numpy()
    n = len(E0)
    E = np.zeros_like(E0)
    for i in range(n):
        for j in range(i + 1, n):
            p = Q[E0[i, j]]
            E[i, j] = E[j, i] = rng.choice(5, p=p / p.sum())
    return E


cat_mol = Chem.MolFromSmiles(ORTHO)
cat_symbols = [a.GetSymbol() for a in cat_mol.GetAtoms()]
cat_pos, cat_E0 = layout(cat_mol), bond_matrix(cat_mol)
rng = np.random.default_rng(3)
steps = [0, 50, 100, 200, 300, 400, 500]
fig, axes = plt.subplots(1, len(steps), figsize=(12.5, 2.0))
for ax, t in zip(axes, steps):
    draw_graph(ax, cat_symbols, noised(cat_E0, t, rng) if t else cat_E0, cat_pos,
               f't = {t},  ᾱ = {alpha_bar[t]:.2f}')
fig.suptitle('Forward process on catechol 2TMS: atoms stay, bonds are resampled from the training marginals',
             x=0.01, ha='left', fontsize=10, fontweight='bold')
axes[0].legend(loc='lower left', bbox_to_anchor=(-0.05, -0.2), ncol=3, handletextpad=0.1, columnspacing=0.6)
plt.tight_layout()
plt.show()

## [s5_reverse]

In [ ]:
@torch.no_grad()
def sample_chain(example, y, n, seed, keep_chain=False):
    """sample_batch from src/diffusion_model_spec2mol.py for n copies of one spectrum, keeping the path if asked."""
    from torch_geometric.data import Batch
    from src import utils as diffms_utils
    from src.diffusion import diffusion_utils

    model = live()['model']
    torch.manual_seed(seed)
    data = Batch.from_data_list([example] * n)
    dense, node_mask = diffms_utils.to_dense(data.x, data.edge_index, data.edge_attr, data.batch)
    z_T = diffusion_utils.sample_discrete_feature_noise(limit_dist=model.limit_dist, node_mask=node_mask)
    X, E, y_batch = dense.X, z_T.E, y.expand(n, -1)
    chain = [E.argmax(-1).numpy().astype(np.uint8)] if keep_chain else None
    for s_int in reversed(range(T)):
        s = s_int * torch.ones((n, 1))
        out, _ = model.sample_p_zs_given_zt(s / T, (s + 1) / T, X, E, y_batch, node_mask)
        E = out.E
        if keep_chain:
            chain.append(E.argmax(-1).numpy().astype(np.uint8))
    nodes, adj = X.argmax(-1), E.argmax(-1)
    smiles = []
    for nd, ad in zip(nodes, adj):
        mol = model.visualization_tools.mol_from_graphs(nd, ad)
        smiles.append(Chem.MolToSmiles(mol) if diffms_utils.is_valid(mol) else None)
    return {'smiles': smiles, 'chain': None if chain is None else np.stack(chain),
            'symbols': [model.dataset_info.atom_decoder[int(i)] for i in nodes[0]]}


def example_of(uid):
    return spectrum_batch(uid)['graph'].get_example(0)


def run_trajectory():
    y = torch.from_numpy(y_of[CATECHOL])[None]
    return sample_chain(example_of(CATECHOL), y, LIVE_BATCH, SEED, keep_chain=True)


traj = cached('trajectory_catechol', run_trajectory)
chain, traj_smiles, symbols = traj['chain'], traj['smiles'], traj['symbols']   # chain: step x sample x n x n; chain[0] is t=T
print(f'{LIVE_BATCH} samples from the catechol spectrum:')
print(pd.Series([s if s else '(invalid)' for s in traj_smiles]).value_counts().to_string())

ortho_canon = Chem.CanonSmiles(ORTHO)
pick = next((i for i, s in enumerate(traj_smiles) if s == ortho_canon),
            next(i for i, s in enumerate(traj_smiles) if s))
final_mol = Chem.MolFromSmiles(traj_smiles[pick])
# Positions come from the final molecule, keyed by node index of the generated graph
final_rw = Chem.RWMol()
for sym in symbols:
    final_rw.AddAtom(Chem.Atom(sym))
for i, j in zip(*np.triu_indices(len(symbols), 1)):
    c = chain[-1, pick, i, j]
    if c:
        final_rw.AddBond(int(i), int(j), {1: Chem.BondType.SINGLE, 2: Chem.BondType.DOUBLE,
                                          3: Chem.BondType.TRIPLE, 4: Chem.BondType.AROMATIC}[int(c)])
gen_mol = final_rw.GetMol()
Chem.SanitizeMol(gen_mol)
gen_pos = layout(gen_mol)

show_t = [500, 200, 100, 50, 25, 10, 0]
fig, axes = plt.subplots(1, len(show_t), figsize=(12.5, 2.0))
for ax, t in zip(axes, show_t):
    draw_graph(ax, symbols, chain[T - t, pick], gen_pos, f't = {t}')
fig.suptitle('Reverse process: one sample conditioned on the catechol spectrum',
             x=0.01, ha='left', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

# When does each sample commit? Fraction of atom pairs differing from its own final graph.
iu = np.triu_indices(len(symbols), 1)
diff = (chain[:, :, iu[0], iu[1]] != chain[-1:, :, iu[0], iu[1]])
n_bonds = (chain[:, :, iu[0], iu[1]] > 0).sum(-1)
tt = T - np.arange(T + 1)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 2.8))
for s in range(LIVE_BATCH):
    axes[0].plot(tt, diff[:, s].mean(-1) * 100, color=C1, alpha=0.25, linewidth=0.8)
axes[0].plot(tt, diff.mean((1, 2)) * 100, color=C1, linewidth=2)
axes[0].set(xlabel='diffusion step t (sampling runs right to left)', ylabel='Atom pairs differing\nfrom the final graph (%)')
axes[0].set_title('Commitment to the final structure')
for s in range(LIVE_BATCH):
    axes[1].plot(tt, n_bonds[:, s], color=C1, alpha=0.25, linewidth=0.8)
axes[1].plot(tt, n_bonds.mean(1), color=C1, linewidth=2)
axes[1].axhline(cat_mol.GetNumBonds(), color=NEUTRAL, linewidth=1)
axes[1].text(T, cat_mol.GetNumBonds() + 0.5, f'{cat_mol.GetNumBonds()} bonds in the true molecule', ha='right',
             va='bottom', fontsize=8, color=MUTED)
axes[1].set(xlabel='diffusion step t', ylabel='Bonds in the graph')
axes[1].set_title('Graph density')
for ax in axes:
    ax.invert_xaxis()
    ax.grid(**GRID_KW)
plt.tight_layout()
plt.show()
for lvl in (50, 25, 10, 5):
    t_half = tt[np.argmax(diff.mean((1, 2)) * 100 <= lvl)]
    print(f'pairs differing from the final graph fall to {lvl}% by t = {t_half}')

## [s5_tail]

## [s6_head]

In [ ]:
def isomer_shares(uid):
    """Share of the 100 samples per isomer. Everything else, invalid samples included, is 'other'."""
    g = by_uid.get(uid)
    out = {k: (0 if g is None else int(g.loc[g['smiles'] == Chem.CanonSmiles(s), 'count'].sum())) for k, s in ISOMERS.items()}
    out['other or invalid'] = N_SAMPLES - sum(out.values())
    return out


same_formula = labels[labels['formula'] == EXAMPLE_FORMULA]
kind = lambda n: ('catechol' if re.search(r'catechol|1,2-diol', n, re.I) else
                  'hydroquinone' if re.search(r'hydroquinone|1,4-diol', n, re.I) else
                  'resorcinol' if re.search(r'1,3-diol', n, re.I) else 'unnamed')
iso = pd.DataFrame([{'uid': u, 'library name': labels.loc[u, 'compound_name'], 'campaign': labels.loc[u, 'source'],
                     'kind': kind(labels.loc[u, 'compound_name']),
                     'base peak': int(raw_spectrum(u)[0][raw_spectrum(u)[1].argmax()]),
                     **isomer_shares(u)} for u in same_formula.index])
iso['_o'] = iso['kind'].map({'catechol': 0, 'unnamed': 1, 'hydroquinone': 2, 'resorcinol': 3})
iso = iso.sort_values(['_o', 'uid']).drop(columns='_o').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9.8, 3.5))
cats, colors = list(ISOMERS) + ['other or invalid'], [C1, C2, C3, NEUTRAL]
ypos = np.arange(len(iso))
left = np.zeros(len(iso))
for c, col in zip(cats, colors):
    ax.barh(ypos, iso[c], left=left, height=0.62, color=col, edgecolor='white', linewidth=1.5, label=c)
    for y_, l_, v in zip(ypos, left, iso[c]):
        if v >= 8:
            ax.text(l_ + v / 2, y_, f'{v}', ha='center', va='center', fontsize=8.5, color='white', fontweight='bold')
    left += iso[c].to_numpy()
ax.set_yticks(ypos, [f"{r['library name']}  ({r['campaign']}, base peak {r['base peak']})" for _, r in iso.iterrows()])
ax.invert_yaxis()
ax.set(xlim=(0, 100), xlabel='Samples out of 100')
ax.set_title(f'Every {EXAMPLE_FORMULA} spectrum in the library: what the model samples')
ax.tick_params(axis='y', length=0)
ax.legend(loc='upper center', bbox_to_anchor=(0.32, -0.16), ncol=4)
plt.tight_layout()
plt.show()

In [ ]:
def grid_for(uid, k=8):
    """Top-k candidates of a spectrum, ranked by count; the reference, if known, drawn first."""
    g = by_uid.get(uid)
    mols, legends = [], []
    true_key = None
    if uid in refs.index:
        true_key = key_2d(refs.loc[uid, 'inchi'])
        mols.append(Chem.MolFromInchi(refs.loc[uid, 'inchi']))
        legends.append('REFERENCE')
    if g is not None:
        for rank, r in enumerate(g.head(k).itertuples(), 1):
            mols.append(Chem.MolFromSmiles(r.smiles))
            flag = ' = reference' if r.key_2d == true_key else (' [wrong H]' if not r.formula_match else '')
            legends.append(f'#{rank}  {r.count}/{N_SAMPLES}{flag}')
    return Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(250, 190), legends=legends)


print('catechol 2TMS (LDY-20562)')
display(grid_for(CATECHOL, k=4))
print('hydroquinone 2TMS (LDY-20550)')
display(grid_for(HYDROQUINONE, k=4))

## [s6_morph_head]

In [ ]:
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
N_MORPH = 48          # samples per condition, 3 batches of LIVE_BATCH
UNRELATED = 'LDY-20949'   # heptadecanoic acid TMS: a spectrum with nothing in common with the example


def mixed_tokens(alpha):
    """Tokens of (1 - alpha) * catechol + alpha * hydroquinone, mixed on the measured intensity scale.

    Same parent formula, so one m/z maps to one token in both spectra. The pipeline's steps are then redone:
    normalise to the base peak, square root, keep the 50 most intense peaks.
    """
    b, forms_a, inten_a = tokens_of(CATECHOL)
    _, forms_b, inten_b = tokens_of(HYDROQUINONE)
    mz_a, mz_b = peak_table(CATECHOL)['mz'].to_numpy(), peak_table(HYDROQUINONE)['mz'].to_numpy()
    form_of, I = {}, Counter()
    for mz, f, i in zip(mz_a, forms_a, inten_a):
        form_of[mz] = f
        I[mz] += (1 - alpha) * float(i) ** 2
    for mz, f, i in zip(mz_b, forms_b, inten_b):
        assert mz not in form_of or torch.equal(form_of[mz], f), f'token differs at m/z {mz}'
        form_of[mz] = f
        I[mz] += alpha * float(i) ** 2
    top = [mz for mz, v in I.most_common(50) if v > 0]
    peak = max(I[mz] for mz in top)
    return b, torch.stack([form_of[mz] for mz in top]), torch.tensor([(I[mz] / peak) ** 0.5 for mz in top], dtype=torch.float32)


def run_condition(y, seed):
    ex = example_of(CATECHOL)
    return sum((sample_chain(ex, y, LIVE_BATCH, seed + i)['smiles'] for i in range(N_MORPH // LIVE_BATCH)), [])


def morph_y(alpha):
    b, f, i = mixed_tokens(alpha)
    return encode(batch_from_tokens(b, f, i))


morph = {a: cached(f'morph_{a:.2f}', lambda a=a: run_condition(morph_y(a), SEED + int(a * 100))) for a in ALPHAS}
controls = {
    'no spectrum (mean y of all 730)': cached('control_mean_y', lambda: run_condition(torch.from_numpy(Y_MEAN)[None], SEED + 200)),
    'unrelated spectrum (C17 acid TMS)': cached('control_unrelated_y', lambda: run_condition(torch.from_numpy(y_of[UNRELATED])[None], SEED + 300)),
}


def tally(smiles):
    c = Counter(s for s in smiles)
    out = {k: c[Chem.CanonSmiles(v)] for k, v in ISOMERS.items()}
    out['other or invalid'] = len(smiles) - sum(out.values())
    return out


rows = pd.DataFrame({a: tally(morph[a]) for a in ALPHAS}).T
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), gridspec_kw={'width_ratios': [1.45, 1]})
ax = axes[0]
for c, col, mark in zip(cats, colors, ('o', 's', '^', 'D')):
    p = rows[c] / N_MORPH * 100
    lo_hi = np.array([wilson(k, N_MORPH) for k in rows[c]]) * 100
    ax.errorbar(rows.index, p, yerr=[p - lo_hi[:, 0], lo_hi[:, 1] - p], color=col, marker=mark, markersize=7,
                markeredgecolor='white', linewidth=2, capsize=2.5, elinewidth=0.9, label=c)
ax.set(xlabel='Hydroquinone fraction of the mixed spectrum', ylabel=f'Samples (%, n = {N_MORPH} each, 95% CI)', ylim=(0, 100))
ax.set_xticks(ALPHAS, ['0\ncatechol', '0.25', '0.5', '0.75', '1\nhydroquinone'])
ax.set_title('Changing only the spectrum moves the sampled isomers')
ax.grid(axis='y', **GRID_KW)
ax.legend(loc='upper center', ncol=2)

ax = axes[1]
ctl = pd.DataFrame({k: tally(v) for k, v in controls.items()}).T
left = np.zeros(len(ctl))
ypos = np.arange(len(ctl))
for c, col in zip(cats, colors):
    ax.barh(ypos, ctl[c] / N_MORPH * 100, left=left, height=0.55, color=col, edgecolor='white', linewidth=1.5)
    left += (ctl[c] / N_MORPH * 100).to_numpy()
ax.set_yticks(ypos, ['no spectrum\n(mean y)', 'unrelated\n(C17 acid TMS)'])
ax.invert_yaxis()
ax.set(xlim=(0, 100), xlabel='Samples (%)')
ax.set_title('Same atoms, spectrum removed')
ax.tick_params(axis='y', length=0)
plt.tight_layout()
plt.show()
display(rows.assign(valid=[sum(s is not None for s in morph[a]) for a in ALPHAS]))
display(ctl.assign(valid=[sum(s is not None for s in v) for v in controls.values()]))
for k, v in controls.items():
    top = Counter(s for s in v if s).most_common(3)
    print(k, '-> most sampled:', top)
print('alpha=0 y vs stored catechol y, max abs difference:',
      f'{np.abs(morph_y(0.0)[0].numpy() - y_of[CATECHOL]).max():.1e}' if _live else 'not recomputed (loaded from cache)')

## [s6_tail]

## [s7_head]

In [ ]:
lab_rows = []
for uid, r in refs.iterrows():
    true_mol = Chem.MolFromInchi(r['inchi'])
    assert CalcMolFormula(true_mol) == r['formula'], uid
    key = key_2d(r['inchi'])
    g = by_uid.get(uid)
    keys = [] if g is None else g['key_2d'].tolist()
    rank = keys.index(key) + 1 if key in keys else np.nan
    fp_true = morgan(true_mol)
    top1 = None if g is None else Chem.MolFromSmiles(g['smiles'].iloc[0])
    lab_rows.append({'uid': uid, 'name': labels.loc[uid, 'compound_name'], 'formula': r['formula'], 'key': key,
                     'smiles': Chem.MolToSmiles(true_mol), 'rank': rank,
                     'count_true': 0 if rank != rank else int(g['count'].iloc[int(rank) - 1]),
                     'top_count': 0 if g is None else int(g['count'].iloc[0]),
                     'n_unique': 0 if g is None else len(g),
                     'validity': summary.loc[uid, 'validity_rate'],
                     'heavy': true_mol.GetNumHeavyAtoms(),
                     'nSi': sum(a.GetSymbol() == 'Si' for a in true_mol.GetAtoms()),
                     'tanimoto_top1': 0.0 if top1 is None else DataStructs.TanimotoSimilarity(morgan(top1), fp_true),
                     'top1_formula_ok': False if g is None else bool(g['formula_match'].iloc[0])})
lab = pd.DataFrame(lab_rows).set_index('uid')

# Seen = the same connectivity (first InChIKey block) is in the training split, with a simulated spectrum.
train_blocks = set(_train['inchikey'].str[:14])
lab['seen_in_training'] = lab.index.map(lambda u: refs.loc[u, 'inchikey'][:14] in train_blocks)


def ranked_keys(c):
    return {u: g['key_2d'].tolist() for u, g in c.groupby('uid', sort=False)}


def pool(a, b):
    """Candidates of both models by summed count, interleaved so that ties favour neither (as in inspect_yee_candidates)."""
    out = {}
    for u in lab.index:
        counts = Counter()
        ga = a[a['uid'] == u]
        gb = b[b['uid'] == u]
        for pair in zip_longest(zip(ga['key_2d'], ga['count']), zip(gb['key_2d'], gb['count'])):
            for p in pair:
                if p is not None:
                    counts[p[0]] += p[1]
        out[u] = [k for k, _ in counts.most_common()]
    return out


def topk_curve(rk):
    return [np.mean([lab.loc[u, 'key'] in rk.get(u, [])[:k] for u in lab.index]) for k in TOP_K_LIST]


rk_tms, rk_notms, rk_pool = ranked_keys(cand), ranked_keys(cand_notms), pool(cand, cand_notms)
curves = {'TMS model': topk_curve(rk_tms), 'Non-TMS model': topk_curve(rk_notms), 'Both pooled': topk_curve(rk_pool)}

fig, ax = plt.subplots(figsize=(6.6, 3.5))
for (name_, c), col, mark in zip(curves.items(), (C1, C2, C3), ('o', 's', '^')):
    ax.plot(TOP_K_LIST, np.array(c) * 100, color=col, marker=mark, markersize=6, markeredgecolor='white', linewidth=2, label=name_)
    ax.text(TOP_K_LIST[-1] * 1.08, c[-1] * 100, name_, va='center', fontsize=8.5, color=INK)
ax.set_xscale('log')
ax.set_xticks(TOP_K_LIST, TOP_K_LIST)
ax.minorticks_off()
ax.set(xlabel='k (candidates ranked by generation count)', ylabel='Reference in top k (%, 2D match)', ylim=(0, 60), xlim=(0.9, 320))
ax.set_title(f'Recovery on {len(lab)} real spectra with a known structure')
ax.grid(**GRID_KW)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()
display(pd.DataFrame(curves, index=[f'top-{k}' for k in TOP_K_LIST]).T.round(3))
print(f'{lab["key"].nunique()} unique 2D structures among {len(lab)} spectra; mean rank-1 Tanimoto {lab["tanimoto_top1"].mean():.2f}')

In [ ]:
# Does training overlap explain the score? And how far is real from simulated?
def seen_row(x):
    n = len(x)
    k1, k10 = int((x['rank'] == 1).sum()), int((x['rank'] <= 10).sum())
    return pd.Series({'spectra': n, 'structures': x['key'].nunique(), 'top-1': k1 / n, 'top-1 95% CI': '%.2f-%.2f' % wilson(k1, n),
                      'top-10': k10 / n, 'top-10 95% CI': '%.2f-%.2f' % wilson(k10, n),
                      'rank-1 Tanimoto': x['tanimoto_top1'].mean()})


display(pd.DataFrame({'not in training': seen_row(lab[~lab['seen_in_training']]),
                      'in training': seen_row(lab[lab['seen_in_training']])}).T.round(3))

# Exp4 test set (non-TMS model, simulated spectra), from feature_difficulty_analysis_ddp.ipynb, section 2.
# Exact InChI match there, so exact InChI here too; those molecules carry no stereo.
EXP4_SIM_TEST = {1: 0.2345, 3: 0.3622, 5: 0.4143, 10: 0.4758, 20: 0.5182, 50: 0.5481, 100: 0.5581}
_exact = {u: g['inchi'].tolist() for u, g in cand_notms.groupby('uid', sort=False)}
real_all = [np.mean([Chem.MolToInchi(Chem.MolFromInchi(refs.loc[u, 'inchi'])) in _exact.get(u, [])[:k] for u in lab.index]) for k in TOP_K_LIST]
no_si = lab.index[lab['nSi'] == 0]
real_no_si = [np.mean([Chem.MolToInchi(Chem.MolFromInchi(refs.loc[u, 'inchi'])) in _exact.get(u, [])[:k] for u in no_si]) for k in TOP_K_LIST]
real_no_si_2d = [np.mean([lab.loc[u, 'key'] in rk_notms.get(u, [])[:k] for u in no_si]) for k in TOP_K_LIST]
display(pd.DataFrame({'simulated test set (n = 7,535)': [EXP4_SIM_TEST[k] for k in TOP_K_LIST],
                      f'real, all references (n = {len(lab)})': real_all,
                      f'real, no Si in structure (n = {len(no_si)})': real_no_si,
                      f'same, 2D match': real_no_si_2d},
                     index=[f'top-{k}' for k in TOP_K_LIST]).T.round(3))

## [s7_tail]

## [s8_head]

In [ ]:
BINS = [(1, 5), (6, 10), (11, 25), (26, 50), (51, 100)]
all_top = cand.groupby('uid', sort=False)['count'].first()
rel = []
for lo, hi in BINS:
    x = lab[(lab['top_count'] >= lo) & (lab['top_count'] <= hi)]
    k = int((x['rank'] == 1).sum())
    rel.append({'rank-1 count': f'{lo}-{hi}', 'labelled spectra': len(x), 'rank-1 correct': k, 'rate': k / max(len(x), 1),
                'lo': wilson(k, len(x))[0], 'hi': wilson(k, len(x))[1], 'rank-1 Tanimoto': x['tanimoto_top1'].mean(),
                'all spectra in the run': int(((all_top >= lo) & (all_top <= hi)).sum())})
rel = pd.DataFrame(rel)

fig, ax = plt.subplots(figsize=(6.8, 3.2))
xp = np.arange(len(rel))
ax.bar(xp, rel['rate'] * 100, width=0.55, color=C1)
ax.errorbar(xp, rel['rate'] * 100, yerr=[(rel['rate'] - rel['lo']) * 100, (rel['hi'] - rel['rate']) * 100], fmt='none',
            ecolor=INK, elinewidth=1, capsize=3)
for i, r in rel.iterrows():
    ax.text(i, r['hi'] * 100 + 2.5, f"{r['rank-1 correct']}/{r['labelled spectra']}", ha='center', fontsize=8.5, color=INK)
ax.set_xticks(xp, rel['rank-1 count'])
ax.set(xlabel=f'How many of the {N_SAMPLES} samples gave the rank-1 structure', ylabel='Rank-1 equals the reference (%)', ylim=(0, 118))
ax.set_yticks([0, 25, 50, 75, 100])
ax.set_title('Sampling frequency is the usable confidence signal')
ax.grid(axis='y', **GRID_KW)
plt.tight_layout()
plt.show()
display(rel.drop(columns=['lo', 'hi']).round(3).style.hide(axis='index'))

# Competing signal: cos(y, candidate fingerprint), as in inspect_yee_candidates.ipynb
by_cos = {u: g.sort_values('cos_fp', ascending=False, kind='stable')['key_2d'].tolist() for u, g in by_uid.items()}
for k in (1, 10):
    n_cnt = sum(lab.loc[u, 'key'] in rk_tms.get(u, [])[:k] for u in lab.index)
    n_cos = sum(lab.loc[u, 'key'] in by_cos.get(u, [])[:k] for u in lab.index)
    print(f'top-{k}: ranking by count finds {n_cnt}/{len(lab)}, by cos(y, fingerprint) {n_cos}/{len(lab)}')

In [ ]:
# The spectra with the highest count that the reference table does not cover: a harder test of the signal above.
top_rows = cand.groupby('uid', sort=False).head(1).set_index('uid')
high = top_rows[top_rows['count'] > 50].copy()
high['library name'] = labels['compound_name'].reindex(high.index).str.replace('\n', ' ').str.slice(0, 44)
high['has reference'] = high.index.isin(lab.index)
high['correct by reference'] = [lab.loc[u, 'rank'] == 1 if u in lab.index else None for u in high.index]
pd.set_option('display.max_colwidth', 60)
display(high.sort_values('count', ascending=False)[['library name', 'formula', 'smiles', 'count', 'formula_match', 'has reference', 'correct by reference']])

## [s8_tail]

## [s9_head]

In [ ]:
motif_q_all = {**MOTIF_Q, 'siloxane Si-O-Si': Chem.MolFromSmarts('[Si]O[Si]')}
rows = []
ref_mols = {u: Chem.MolFromSmiles(s) for u, s in lab['smiles'].items()}
top1_mols = {u: Chem.MolFromSmiles(top_rows.loc[u, 'smiles']) for u in lab.index if u in top_rows.index}
for name_, q in motif_q_all.items():
    has_true = pd.Series({u: m.HasSubstructMatch(q) for u, m in ref_mols.items()})
    has_top1 = pd.Series({u: top1_mols[u].HasSubstructMatch(q) if u in top1_mols else False for u in ref_mols})
    tp, fn, fp = int((has_true & has_top1).sum()), int((has_true & ~has_top1).sum()), int((~has_true & has_top1).sum())
    x = lab[has_true.reindex(lab.index)]
    rows.append({'motif': name_, 'spectra with it': int(has_true.sum()),
                 'recall in rank-1': tp / max(tp + fn, 1), 'precision in rank-1': tp / max(tp + fp, 1),
                 'top-1 accuracy': (x['rank'] == 1).mean(), 'top-10 accuracy': (x['rank'] <= 10).mean()})
x = lab[lab['nSi'] == 0]
rows.append({'motif': 'no Si at all', 'spectra with it': len(x), 'top-1 accuracy': (x['rank'] == 1).mean(), 'top-10 accuracy': (x['rank'] <= 10).mean()})
display(pd.DataFrame(rows).round(2).style.hide(axis='index').format(na_rep='-', precision=2))

oosi = Chem.MolFromSmarts('[OX2]([OX2])[Si]')
print(f'rank-1 candidates over all spectra with an O-O-Si group: '
      f'{np.mean([Chem.MolFromSmiles(s).HasSubstructMatch(oosi) for s in top_rows["smiles"]]):.0%}')
print(f'rank-1 candidates with the wrong H count: labelled {1 - lab["top1_formula_ok"].mean():.0%}, '
      f'all spectra {1 - top_rows["formula_match"].mean():.0%}')
print('mean validity by heavy atoms (all labelled):')
print(lab.groupby(pd.cut(lab['heavy'], [0, 15, 20, 25, 30, 60]))['validity'].mean().round(2).to_string())
_all_h = summary.join(pd.Series({u: sum(int(n or 1) for e, n in re.findall(r'([A-Z][a-z]?)(\d*)', f) if e != 'H')
                                 for u, f in summary['formula'].items()}, name='heavy'))
print(f'spectra with no valid candidate: {(summary["num_valid"] == 0).sum()} of {len(summary)}, '
      f'median {_all_h.loc[_all_h["num_valid"] == 0, "heavy"].median():.0f} heavy atoms; those with C and H only: '
      f'{sum(set(re.findall("[A-Z][a-z]?", f)) <= {"C", "H"} for f in summary.loc[summary["num_valid"] == 0, "formula"])}')

In [ ]:
def versus(uids):
    """Reference above rank-1 candidate, with the maximum common substructure highlighted in the candidate."""
    mols, legends, highlight = [], [], []
    for u in uids:
        t, c = ref_mols[u], top1_mols[u]
        mcs = rdFMCS.FindMCS([t, c], timeout=3, ringMatchesRingOnly=True, completeRingsOnly=True)
        patt = Chem.MolFromSmarts(mcs.smartsString)
        r = lab.loc[u]
        short = re.sub(r'\s+', ' ', str(r['name']))[:30]
        rank_txt = 'not generated' if r['rank'] != r['rank'] else f"rank {int(r['rank'])}"
        mols.extend([t, c])
        legends.extend([f'{short} ({rank_txt})',
                        f"rank-1 {r['top_count']}/{N_SAMPLES}, Tanimoto {r['tanimoto_top1']:.2f}"
                        + ('' if r['top1_formula_ok'] else ', wrong H')])
        highlight.extend([[], list(c.GetSubstructMatch(patt))])
    # Molecule pairs read top to bottom, so transpose the column order into the row-wise grid
    n = len(uids)
    order = list(range(0, 2 * n, 2)) + list(range(1, 2 * n, 2))
    return Draw.MolsToGridImage([mols[i] for i in order], molsPerRow=n, subImgSize=(280, 210),
                                legends=[legends[i] for i in order], highlightAtomLists=[highlight[i] for i in order])


GOOD = ['LDY-16805', 'LDY-20796', 'LDY-20828', 'LDY-20949']   # 3,4-DHBA 3TMS, erythritol 4TMS, arabinopyranose 4TMS, C17 acid TMS
NEAR = ['LDY-19729', 'LDY-16717', 'LDY-18973', 'LDY-18928']   # terephthalic acid 2TMS, eugenol TMS, C16 acid TMS, a pentitol 5TMS
BAD  = ['LDY-20761', 'LDY-20948', 'LDY-20868', 'LDY-21096']   # succinic acid 2TMS, malic acid 3TMS, a branched FAME, nicotine
print('Recovered: top row is the reference, bottom row the rank-1 candidate')
display(versus(GOOD))
print('Close: right class, wrong position, chain or group')
display(versus(NEAR))
print('Wrong: the silyl ester blind spot, and compounds without Si')
display(versus(BAD))

## [s9_tail]

## [s10_head]

In [ ]:
named_tentative = refs.index
unknown = top_rows[(~labels['identified'].reindex(top_rows.index).fillna(False).astype(bool))
                   & (top_rows['count'] >= 26) & top_rows['formula_match'] & ~top_rows.index.isin(named_tentative)].copy()
unknown = unknown.sort_values('count', ascending=False)
unknown['library code'] = labels['compound_name'].reindex(unknown.index)
unknown['campaign'] = labels['source'].reindex(unknown.index)
unknown['retention index'] = labels['retention_index'].reindex(unknown.index)
display(unknown[['library code', 'campaign', 'formula', 'smiles', 'count', 'retention index']])
display(Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in unknown['smiles']], molsPerRow=4, subImgSize=(260, 190),
                             legends=[f'{u}  {r.formula}  {r.count}/{N_SAMPLES}' for u, r in unknown.iterrows()]))

## [s10_tail]

## [s11_head]

In [ ]:
def show_case(uid, k=9):
    """Spectrum, reference if there is one, and the top candidates."""
    name = re.sub(r'\s+', ' ', str(labels.loc[uid, 'compound_name']))
    s = summary.loc[uid]
    print(f"{uid}  {name}  |  {labels.loc[uid, 'formula']}  |  {labels.loc[uid, 'source']}  |  "
          f"{s['num_valid']}/{s['num_samples']} valid, {s['unique_valid']} unique")
    mz, inten = raw_spectrum(uid)
    kept = np.isin(mz, peak_table(uid)['mz'])
    fig, ax = plt.subplots(figsize=(10, 2.6))
    ax.vlines(mz[~kept], 0, inten[~kept] * 100, color=NEUTRAL, linewidth=0.8)
    ax.vlines(mz[kept], 0, inten[kept] * 100, color=C1, linewidth=1.1)
    for i in np.argsort(-inten)[:7]:
        ax.text(mz[i], inten[i] * 100 + 2, f'{mz[i]:.0f}', ha='center', fontsize=8, color=INK)
    ax.set(xlabel='m/z', ylabel='Relative intensity (%)', ylim=(0, 115))
    ax.grid(axis='y', **GRID_KW)
    plt.tight_layout()
    plt.show()
    display(grid_for(uid, k=k))


try:
    import ipywidgets as widgets
    _options = sorted(((re.sub(r'\s+', ' ', str(labels.loc[u, 'compound_name']))[:46] + f'  [{u}]', u) for u in names),
                      key=lambda p: (p[1] not in lab.index, p[0]))
    display(widgets.interactive(show_case, uid=widgets.Dropdown(options=_options, value=CATECHOL, description='spectrum',
                                                                layout=widgets.Layout(width='620px')),
                                k=widgets.IntSlider(value=9, min=1, max=19, description='candidates')))
except ImportError:
    print("ipywidgets not installed: call show_case('LDY-####') directly.")

In [ ]:
show_case('LDY-16805')

## [s12]